# 🛡️ Real-Time Fraud Detection & Risk Intelligence
## Notebook 01: Exploratory Data Analysis (EDA) & Imbalance Analysis

This notebook provides exploratory analysis on the credit card transactions dataset, detailing:
- Severe class imbalance (~0.17% fraud rate)
- Amount and Time feature distributions
- PCA feature correlations with transaction fraud
- Strategies for leakage prevention and metric selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.ingestion import DataIngestion

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

### 1. Load Dataset and Inspect Schema

In [ ]:
ingestion = DataIngestion()
df = ingestion.load_data()
print(f'Total Rows: {df.shape[0]:,}, Total Columns: {df.shape[1]}')
display(df.head())

### 2. Class Imbalance: Legit (0) vs Fraud (1)

In [ ]:
counts = df['Class'].value_counts()
percentages = df['Class'].value_counts(normalize=True) * 100
print(f'Legitimate (Class 0): {counts[0]:,} ({percentages[0]:.3f}%)')
print(f'Fraudulent (Class 1): {counts[1]:,} ({percentages[1]:.3f}%)')
print(f'Imbalance Ratio: 1 fraud for every {counts[0]//counts[1]:,} legitimate transactions')

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='Class', data=df, ax=ax, palette=['#43a047', '#e53935'])
ax.set_yscale('log')
ax.set_title('Transaction Distribution (Log Scale)')
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
plt.tight_layout()
plt.show()

### 3. Transaction Amount Distribution

In [ ]:
print('Amount Summary for Legitimate Transactions:')
print(df[df['Class'] == 0]['Amount'].describe())

print('\nAmount Summary for Fraudulent Transactions:')
print(df[df['Class'] == 1]['Amount'].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df['Class'] == 0]['Amount'], bins=50, kde=True, ax=ax1, color='#43a047')
ax1.set_title('Legit Amount Distribution')
ax1.set_xlim(0, 1000)

sns.histplot(df[df['Class'] == 1]['Amount'], bins=50, kde=True, ax=ax2, color='#e53935')
ax2.set_title('Fraud Amount Distribution')
ax2.set_xlim(0, 1000)
plt.tight_layout()
plt.show()

### 4. Correlation with Target Class

In [ ]:
corr = df.corr()['Class'].drop('Class').sort_values()
print('Top 5 Inversely Correlated Features (Lower value -> Higher Fraud Risk):')
print(corr.head(5))
print('\nTop 5 Positively Correlated Features (Higher value -> Higher Fraud Risk):')
print(corr.tail(5))

fig, ax = plt.subplots(figsize=(10, 6))
corr.plot(kind='bar', ax=ax, color=np.where(corr>0, '#e53935', '#1e88e5'))
ax.set_title('Correlation of PCA & Engineered Features with Target Class')
plt.tight_layout()
plt.show()